In [0]:
%pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

import nltk
import random
import pandas as pd
from nltk.corpus import movie_reviews
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Download required NLTK data
nltk.download('movie_reviews')



[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/spark-10581cd4-5bb6-47e8-821f-17/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


True

In [0]:
# Prepare movie reviews dataset
documents = [(list(movie_reviews.words(fileid)), category)
            for category in movie_reviews.categories()
            for fileid in movie_reviews.fileids(category)]

# Shuffle and prepare data
random.shuffle(documents)
reviews = [" ".join(words) for words, _ in documents]
sentiments = [sentiment for _, sentiment in documents]

# Create a DataFrame for better visualization
df = pd.DataFrame({
    'text': reviews,
    'sentiment': sentiments
})

print("Dataset Overview:")
print(df['sentiment'].value_counts())
print("\nSample Reviews:")
print(df.head())


Dataset Overview:
sentiment
pos    1000
neg    1000
Name: count, dtype: int64

Sample Reviews:
                                                text sentiment
0  with stars like sigourney weaver ( " alien " t...       pos
1  there ' s a good , and timely , story trapped ...       neg
2  we could paraphrase michelle pfieffer ' s char...       neg
3  this is my first review that i post to this ne...       neg
4  set in the late 80 ' s at the height of yuppie...       pos


In [0]:

# Create and configure TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
    stop_words='english'
)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['sentiment'], 
    test_size=0.2, 
    random_state=42
)

# Transform text data
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Display feature information
print("\nNumber of features:", len(vectorizer.get_feature_names_out()))
print("Top features:", vectorizer.get_feature_names_out()[:10])



Number of features: 5000
Top features: ['000' '10' '100' '101' '11' '12' '13' '13th' '14' '15']


In [0]:
# Initialize and train the classifier
classifier = LogisticRegression(random_state=42, max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

# Make predictions
y_pred = classifier.predict(X_test_tfidf)

# Print detailed performance metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Print overall accuracy
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")



Classification Report:
              precision    recall  f1-score   support

         neg       0.86      0.86      0.86       196
         pos       0.87      0.87      0.87       204

    accuracy                           0.86       400
   macro avg       0.86      0.86      0.86       400
weighted avg       0.86      0.86      0.86       400

Accuracy: 0.8650


## The Text Processing Journey (Raw Strings to Numerical Vectors)

The raw text data undergoes a structured pipeline to transform unstructured strings into a numerical sparse matrix ready for machine learning:

```
[Raw Text Strings] ➔ [Train/Test Split] ➔ [Tokenization & Stop Word Removal] ➔ [Vocabulary Filtering] ➔ [TF-IDF Weighting] ➔ [Numerical Sparse Matrix]

```

---

### 1. Data Splitting

* **`train_test_split(...)`**: The raw text dataset (`df['text']`) is partitioned into an **80% training set** (`X_train`) and a **20% testing set** (`X_test`), reserving unseen data to evaluate the final model without data leakage.

### 2. Configuration & Vocabulary Filtering (`TfidfVectorizer`)

The `TfidfVectorizer` is configured with hyper-parameters to clean and constrain the vocabulary:

* **Stop Word Removal (`stop_words='english'`)**: Filters out non-informative English words (e.g., *"the"*, *"is"*, *"and"*).
* **Document Frequency Thresholds**:
* **`min_df=5`**: Excludes extremely rare terms appearing in fewer than 5 documents (reduces noise and typos).
* **`max_df=0.7`**: Excludes overly common terms appearing in over 70% of documents (removes ubiquitous corpus-specific words).


* **Feature Cap (`max_features=5000`)**: Keeps only the top 5,000 most frequent terms that pass the frequency thresholds.

### 3. Feature Extraction & Vectorization

* **`fit_transform(X_train)`**:
* **Tokenizes** training text into lowercased tokens.
* Builds the vocabulary dictionary based on frequency limits.
* Calculates **TF-IDF** scores:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$


* **TF (Term Frequency)**: Frequency of term $t$ in document $d$.
* **IDF (Inverse Document Frequency)**: Penalty factor for terms that appear across many documents.


* Outputs `X_train_tfidf` as a numerical sparse matrix of shape $(\text{samples}, 5000)$.


* **`transform(X_test)`**: Applies the exact vocabulary and IDF weights learned during training to transform test documents into vectors without leaking test information into the vectorizer.

---

## Evaluation of Training Setup & Effectiveness

### 1. Strengths of the Pipeline

* **Leak-Free Transformation**: Proper separate execution of `fit_transform` on `X_train` and `transform` on `X_test` prevents train-test data leakage.
* **Effective Model Choice**: Logistic Regression combined with TF-IDF vectors serves as an efficient, highly readable, and strong baseline for sentiment analysis tasks.
* **Controlled Dimensionality**: Limiting vocabulary size to 5,000 and setting document frequency cutoffs (`min_df`/`max_df`) significantly reduces memory overhead and prevents severe overfitting.

---

### 2. Limitations & Areas for Improvement

* **Presence of Numerical Noise**:
* Looking at the printed output:
> `Top features: ['000', '10', '100', '101', '11', '12', '13', '13th', '14', '15']`


* Pure numbers (e.g., `'000'`, `'100'`, `'101'`) occupy valuable feature slots out of the 5,000 limit because no regex/token pattern was configured to strip digits. These numbers rarely carry sentiment information.


* **Lack of Context / Word Order (Unigram Limit)**:
* Default `TfidfVectorizer` uses single words (`ngram_range=(1, 1)`). It misses context from multi-word expressions or negations (e.g., *"not good"* vs. *"good"*). Adding bigrams (`ngram_range=(1, 2)`) would capture negation and local context far better.

In [0]:
# Get feature importance scores
feature_names = vectorizer.get_feature_names_out()
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': abs(classifier.coef_[0])
})

# Sort and display top important features
print("\nTop 10 Most Important Words for Classification:")
print(feature_importance.sort_values('importance', ascending=False).head(10))



Top 10 Most Important Words for Classification:
       feature  importance
382        bad    3.108334
4965     worst    1.875948
3347      plot    1.643966
530     boring    1.625546
1974     great    1.560283
2636      life    1.553240
4364  supposed    1.411305
4304    stupid    1.389034
3611    reason    1.257652
4853     waste    1.244835


In [0]:
# Create DataFrame with predictions
results_df = pd.DataFrame({
    'Text': X_test,
    'Actual': y_test,
    'Predicted': y_pred
})

# Filter and display misclassified examples
print("\nMisclassified Examples:")
misclassified = results_df[results_df['Actual'] != results_df['Predicted']]
print(misclassified.head(3))



Misclassified Examples:
                                                   Text Actual Predicted
746   miramax " disinvited " on - line media from pr...    neg       pos
1664  the long kiss goodnight ( r ) meryl streep tri...    pos       neg
964   i had a chance to see a sneak preview of city ...    pos       neg


---

### 1. `1289`: `"this remake of ' la cage aux folles ' features..."`

* **Actual:** `pos` | **Predicted:** `neg`
* **What it reveals:**
* **Sensitivity to Foreign Words & Title Noise:** Foreign words (like *"la cage aux folles"*) or specific film titles are likely rare in the training set or treated as meaningless noise. If the surrounding sentence uses neutral descriptive language rather than overt positive sentiment words (e.g., *"excellent"*, *"great"*), a bag-of-words model fails to infer positive context.
* **Lack of Semantic Understanding:** A linear TF-IDF model cannot understand the implicit praise often found in movie summaries or comparisons.



---

### 2. `907`: `"absolute power , the new film produced and dir..."`

* **Actual:** `neg` | **Predicted:** `pos`
* **What it reveals:**
* **Swayed by High-Weight Positive-Sounding Words:** The phrase *"absolute power"* contains terms that often carry strong weight or are associated with grandeur/intensity.
* **Inability to Handle Tone & Context:** Words describing plot elements or titles can mislead the classifier. The model assigns positive weights to strong vocabulary without understanding that the reviewer is using those words in a negative or critical critique.



---

### 3. `1290`: `"lean , mean , escapist thrillers are a tough p..."`

* **Actual:** `pos` | **Predicted:** `neg`
* **What it reveals:**
* **Failure with Idioms, Slang, & Complex Nuance:** Words like `"mean"` or `"tough"` traditionally carry **negative sentiment** in standard dictionaries and training corpora. However, in the context of action/thriller films, *"lean, mean"* is a common idiom meaning *tightly written and punchy* (positive), and *"tough"* can describe a good challenge or grit.
* **Unigram Blindness:** Because the vectorizer treats `"lean"`, `"mean"`, and `"tough"` as independent single words rather than recognizing the full phrase `"lean, mean, escapist thriller"`, it sums up the negative individual word weights and incorrectly flips the prediction to `neg`.



---

## Summary of Methodological Weaknesses

| Issue Shown in Examples | Why Your Method Fails | How to Fix It |
| --- | --- | --- |
| **Word Order & Context Blindness** | Unigrams (`ngram_range=(1,1)`) look at words isolated from surrounding context. | Enable n-grams (`ngram_range=(1, 2)`) to capture multi-word expressions like *"lean mean"*. |
| **Misinterpreting Domain-Specific Nuance** | Words like `"mean"` or `"tough"` default to standard negative weights. | Fine-tune sub-word tokenizers or switch to context-aware transformers (e.g., BERT, RoBERTa). |
| **Title & Plot Summary Noise** | Text includes non-sentiment words (titles, character names) that skew classification. | Perform named entity recognition (NER) or custom cleaning to strip out movie titles before vectorization. |